## A scripted agent that asks about your day

This notebook runs a basic conversational agent that asks a user about their day. By default, it asks hour by hour (see below for the other option, ```PART_OF_DAY```). For each hour, the agent asks what the user did and then asks follow-up questions for the details that the user did not mention yet:

* **with_whom**: who else took part
* **what**: what the activity was about, e.g. the food, objects or topics
* **with_what**: the tools, devices or vehicles that were used
* **where**: the place
* **when**: the time at which it started
* **how_long**: how long it took

The dialogue itself is scripted: the agent always asks the same questions in the same order. An LLM served by a local [llama.cpp](https://github.com/ggml-org/llama.cpp) server is only used to interpret the answers of the user: it decides which details the user already gave and fills the slots of the event. After each event, the agent moves on to the next hour after the event ended, until the user types ```stop``` (or presses the STOP button) or the day is over.

The result is saved as a JSON file with the same structure as the examples in ```data/example_days```: an object with ```event1```, ```event2```, ... and the ```coreference_chains``` of the people in the events.

The code of the agent is in ```day_interviewer.py``` in the same folder as this notebook.

### Hour by hour or per part of the day

The agent has two strategies, set with ```PART_OF_DAY``` in the settings:

* **hour by hour** (```PART_OF_DAY = False```, the default): the agent asks about each hour and asks follow-up questions for the details that are missing, as described above. This gives many details per event, but takes many turns: often more than a hundred for a whole day.
* **per part of the day** (```PART_OF_DAY = True```): the agent only asks *What did you do in the morning?*, *... in the afternoon?* and *... in the evening?*. The LLM extracts all the activities from each answer as separate events, in the order in which they happened, using only what the user said. If the user gives no times, the events still get the order that makes sense, e.g. waking up before breakfast, but the start time and duration are left out: ```when``` only has the part of the day, e.g. ```["morning"]```, and ```how_long``` is ```null```. There are no follow-up questions and no photo questions.

The second strategy needs only three answers for a whole day, but the coverage of the roles is poor: whatever the user does not mention stays empty.

This notebook runs the agent **twice**, once with each strategy, so that you can tell about the same day to both. At the end, the statistics of the two interviews are compared side by side: the number of turns, events and roles, how many turns each event and role took, which roles and fillers were collected and the coreference chains. This shows the effect of the two extreme scripted ways of interviewing. Compare the two also with the effectiveness of the communication in the evaluation: how many (correct) events and roles each turn yields. A smarter agent would combine the strengths of both: let the user tell freely and only ask about what is missing.

### Photos

When you say that you have a photo of an activity, e.g. *Here's a photo of the four of us* or *My colleague took a picture of me*, the agent asks you to describe what is on the photo: who and what can be seen, and where it was taken. The agent cannot see the photo itself. The LLM first checks that you mean a photo of the activity, and not e.g. that you looked at photos. From your description, it decides which roles the photo shows and the agent does not ask for these anymore. As in the example days, these roles get the value ```<PHOTO>``` in the event and their values are kept in the ```photo``` of the event:

* ```note```: a sentence about the photo
* ```fills```: the roles that the photo shows, e.g. ```["what", "where", "with_whom"]```
* ```gold```: the values of these roles, e.g. ```{"with_whom": ["his wife Gerda"]}```
* ```people_in_photo```: the people on the photo with their name, role, position and appearance, as far as you described them
* ```file```: ```null```, since the agent does not get the photo file

The people on a photo are also in the coreference chains, with the field ```photo.gold.with_whom```. The statistics and the evaluation read the values of the roles from the ```gold``` of the photo.

## Prerequisites

The agent calls a llama_cpp server that needs to run locally (https://llama-cpp.com).
The installation procedure is described in [llama.cpp](https://github.com/ggml-org/llama.cpp) (e.g. ```brew install llama.cpp``` on macOS).


### Starting the llama.cpp server

After installing llama_cpp, open a terminal launch the server with the Qwen2.7-7B model as follows:

```bash
llama-server -hf Qwen/Qwen2.5-7B-Instruct-GGUF:Q4_K_M --port 8080 -c 4096
```

This will load the Qen model from Huggingface.co. Other models that you can server through llama_cpp can be found here: [huggingface.co](https://huggingface.co/models?library=gguf). The server then provides an OpenAI-compatible API at ```http://localhost:8080/v1```. No API key is needed, but the ```openai``` client requires a value, so any string will do.


### Client modules to make calls to the server

The agent uses the ```openai``` Python client to call the llama.cpp server, ```ipywidgets``` for the chat window and ```pandas``` to show the result:

```
!pip install openai
!pip install ipywidgets
!pip install pandas
```

These packages should have been installed already through the requirements.txt in your virtual environment.


In [1]:
import sys
from pathlib import Path

# Jupyter runs the notebook in its own folder. We add the root of the project to the path,
# so that the agent can be imported from src.cltl.demo_chatbot
PROJECT_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "src" / "cltl").is_dir())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import json
from collections import Counter
import pandas as pd
from openai import OpenAI
from src.cltl.demo_chatbot.day_interviewer import DayInterviewer
from src.cltl.demo_chatbot.chat_window import ChatWindow
from src.cltl.demo_chatbot.day_statistics import ROLES, get_day_statistics, get_events, get_role_values, get_turns

## Settings

* ```LLAMA_CPP_URL```: the address of the llama.cpp server
* ```EXAMPLES```: the folder with the example days. The agent takes the event types from these files and gives a few of the events to the LLM as examples of how to fill the slots.
* ```OUTPUT```: the folder in which the day is saved as ```day_<name>_<strategy>_<date>.json``` (```<strategy>``` is ```hour_by_hour``` or ```part_of_day```), by default ```data/collected_days``` in the root of the project (not in the folder of this notebook). The file is saved after every event, so nothing is lost if the conversation is interrupted. The window shows where the file is saved. Note that the ```data``` folder is in ```.gitignore```, so your days are not committed to git and the folder may be shown in grey in your editor.
* ```FONT_SIZE```: the font size of the chat window in pixels at the start. It can also be changed with the slider at the top of the window.
* ```ASK_ALL```: with ```True```, the agent asks every question for every event, also when the user already gave the answer. With ```False```, the agent only asks for the details that are missing. Only used hour by hour.

In [3]:
LLAMA_CPP_URL = "http://localhost:8080/v1"
EXAMPLES = PROJECT_ROOT / "data" / "example_days"
OUTPUT = PROJECT_ROOT / "data" / "collected_days"
FONT_SIZE = 18
ASK_ALL = False

## Connecting to the llama.cpp server

In [4]:
llama_client = OpenAI(base_url=LLAMA_CPP_URL, api_key="sk-no-key-required")

# llama.cpp serves the model it was started with, the name is only used for reporting
model = llama_client.models.list().data[0].id
print('Model served by llama.cpp:', model)

Model served by llama.cpp: Qwen/Qwen2.5-7B-Instruct-GGUF:Q4_K_M


## Talking to the agent

The agent is called twice: first hour by hour and then per part of the day. Run the next cell to open the chat window of the first interview, finish it with **Stop** and then run the cell after it for the second interview. Tell both agents about the same day, otherwise the comparison at the end says little about the strategies. If you changed ```day_interviewer.py``` or ```chat_window.py```, restart the kernel first, since Python only imports a module once. The questions of the agent and your answers are shown together in the window, with the agent on the left and your answers on the right. Type your answer in the text field below the conversation and press **enter** to send it. Each answer takes a few seconds, since the LLM needs to process it. While the agent is thinking, the text field is disabled.

* **Skip question**: sends an empty answer. When the agent asks what you did at a certain hour or in a part of the day, this skips that hour or part. When the agent asks a follow-up question, e.g. who you were with, the question is left unanswered.
* **Font size**: use the slider at the top of the window to make the text bigger or smaller.
* **Stopping**: click the red **Stop** button or type ```stop```, ```bye```, ```done```, ```quit``` or ```that's all``` as an answer. If you stop in the middle of the follow-up questions of an hour, that event is saved with the answers you gave so far. The agent then groups the people in the coreference chains, saves the day and disables the window. The last message of the agent and the line below the window tell where the file is saved. The conversation also stops by itself when the day is over, after midnight.

The events are saved after each answered hour, so if the kernel is interrupted the events so far are kept, but the file then has no ```coreference_chains```. Always stop the conversation with **Stop** to get the complete file.

### Interview 1: hour by hour

The agent asks about each hour, with follow-up questions for the details that are missing. The parameter ```part_of_day``` is set to False.

In [ ]:
agent_hours = DayInterviewer(llama_client, model, examples_folder=EXAMPLES, output_folder=OUTPUT, ask_all=ASK_ALL,
                             part_of_day=False)
ChatWindow(agent_hours, font_size=FONT_SIZE).show()

### Interview 2: per part of the day

The agent only asks about the morning, the afternoon and the evening, without follow-up questions.

In [ ]:
agent_parts = DayInterviewer(llama_client, model, examples_folder=EXAMPLES, output_folder=OUTPUT, ask_all=ASK_ALL,
                             part_of_day=True)
ChatWindow(agent_parts, font_size=FONT_SIZE).show()

## The result

Run the next cells after both conversations are finished. Each day is saved in the JSON file given at the end of its conversation. We load both again and show the events of each interview as a table. Each row is an event with the slots filled by the LLM. The ```when``` slot is not filled by the LLM but made from the start time, in the same way as in the example days, e.g. ```["05:45", "quarter to six", "6 o'clock", "early in the morning", "morning"]```. With ```PART_OF_DAY = True```, an event of which you gave no time only has the part of the day, e.g. ```["morning"]```, and no ```how_long```.

The next cell uses the ```agent_hours``` and ```agent_parts``` results that have just been created in this notebook. It first checks if these have been saved and the conversations finished. Next it gets the paths for the saved data.

In [ ]:
agents = {"hour by hour": agent_hours, "part of day": agent_parts}
for mode, agent in agents.items():
    if agent.path is None:
        raise ValueError(f"No events saved yet for {mode}, first tell the agent about your day in its chat window.")
    if not agent.done:
        print(f"The {mode} conversation is not finished yet, these are the events so far. Click Stop to finish it.")
day_files = {mode: agent.path for mode, agent in agents.items()}

To compare two days that were saved before, e.g. after restarting the kernel, give their paths in ```day_files``` instead of the paths of the agents.
Only run the next cell to load other files saved before.

In [5]:
day_files = {"hour by hour": OUTPUT / "day_emma_hour_by_hour_20261022_1104.json",
             "part of day": OUTPUT / "day_emma_part_of_day_20261022_1149.json"}


In [6]:
days = {}
for mode, path in day_files.items():
    print(f"The {mode} day is saved in", path)
    with open(path) as json_file:
        days[mode] = json.load(json_file)

The hour by hour day is saved in /Users/piek/Desktop/Combot2026/ma-communicative-robots/data/collected_days/day_emma_hour_by_hour_20261022_1104.json
The part of day day is saved in /Users/piek/Desktop/Combot2026/ma-communicative-robots/data/collected_days/day_emma_part_of_day_20261022_1149.json


The next cell prints a Pandas frame for the data extracted by both interviews, showing each event on a row:

In [7]:
columns = ["event_type", "event_label", "when", "how_long", "where", "what", "with_what", "with_whom", "description"]
for mode, day in days.items():
    print(mode)
    display(pd.DataFrame(get_events(day)).T.reindex(columns=columns))

hour by hour


,event_type,event_label,when,how_long,where,what,with_what,with_whom,description
event1,personal_care,waking up,"[07:00, 7 o'clock, early in the morning, morning]",5,"[bedroom of her house, bedroom, her house, Ams...",[],[alarm clock],[],woke up to the alarm and stayed in bed for a f...
event2,personal_care,showering and getting dressed,"[08:00, 8 o'clock, early in the morning, morning]",20,"[shared bathroom, bathroom, her house, Amsterd...","[shower, clothes]","[shower, clothes]",[],took a shower and got dressed
event3,meal,breakfast,"[09:00, 9 o'clock, in the morning, morning]",45,<PHOTO>,<PHOTO>,<PHOTO>,<PHOTO>,"ate food including yoghurt, fruit, marmalade, ..."
event4,travel,cycling to uni,"[10:00, 10 o'clock, in the morning, morning]",30,"[university, Amsterdam, Netherlands, Europe]",[],[bicycle],[],cycled to the university
event5,study,studying,"[11:00, 11 o'clock, late in the morning, morning]",60,"[library, Amsterdam, Netherlands, Europe]","[books, laptop]","[books, laptop]",[Lotte],studied for her class with Lotte


part of day


,event_type,event_label,when,how_long,where,what,with_what,with_whom,description
event1,personal_care,wake up,"[07:30, half past seven, 7 o'clock, early in t...",None,"[bedroom of her house, her house, Amsterdam, E...",[],[],[],woke up when the alarm went off
event2,personal_care,shower,[morning],None,"[bathroom of her house, bathroom, her house, A...",[],[],[],took a shower
event3,personal_care,get dressed,[morning],None,"[bedroom of her house, her house, Amsterdam, E...",[],[],[],got dressed
event4,meal,breakfast,"[08:00, 8 o'clock, early in the morning, morning]",None,"[kitchen of her house, kitchen, her house, Ams...","[yoghurt, fruit, coffee]","[kitchen of her house, kitchen, her house, Ams...","[Lotte, Suzie]",had yoghurt with fruit and a coffee with Lotte...
event5,travel,bike to university,"[09:00, 9 o'clock, in the morning, morning]",None,"[university, Amsterdam, Europe]",[],[bicycle],[],went to the university on her bike
event6,study,study at the library,"[10:00, 10 o'clock, in the morning, morning]",None,"[library, university, Amsterdam, Europe]","[laptops, readings]",[library],[Lotte],studied together for her class using her lapto...
event7,study,going to class,[afternoon],None,"[classroom, school, Amsterdam, Europe]",[],[],[Lotte],went to class with Lotte
event8,meal,having lunch,[afternoon],None,"[lunch place, cafeteria, Amsterdam, Europe]","[sandwich, coffee]",[],[],had a quick sandwich and more coffee
event9,shopping,doing some shopping,[afternoon],None,"[home, Amsterdam, Europe]",[],[],[],went home to do some shoppings
event10,household,cleaning her room,[afternoon],None,"[bedroom of her house, her house, Amsterdam, E...",[],[],[],cleaned her room a bit


Note the the part-of-day mode has many more events but less information for each event. 

The questions of the agent about each event are kept as a list in ```agent_turn``` and the answers of the user as a list in ```human_turn```. Each turn has the utterance and the number of the turn in the conversation, e.g. ```{"turn_id": 9, "utterance": "What did you do at 7 o'clock?"}```. ```human_turn[i]``` is the answer to ```agent_turn[i]```, with the next ```turn_id```. A skipped question has an empty answer. With ```PART_OF_DAY = True```, all the events extracted from one answer have the same question and answer:

In [8]:
for mode, day in days.items():
    print(f"=== {mode}")
    for key, event in get_events(day).items():
        print(f"{key}:")
        questions = get_turns(event, "agent_turn")
        for i, answer in enumerate(get_turns(event, "human_turn")):
            if i < len(questions):
                print(f"    {questions[i]['turn_id']:>3} Agent:", questions[i]["utterance"])
            print(f"    {answer['turn_id']:>3} You:  ", answer["utterance"] or "(skipped)")

=== hour by hour
event1:
      9 Agent: What did you do at 7 o'clock?
     10 You:   My alarm went off
     11 Agent: Who were you with?
     12 You:   in my bed in my appartment
     13 Agent: What did you use for it, such as tools, devices or a vehicle?
     14 You:   (skipped)
     15 Agent: Where were you?
     16 You:   in my bed in my appartment
     17 Agent: How long did it take?
     18 You:   5 minutes
event2:
     19 Agent: What did you do at 8 o'clock?
     20 You:   I took a shower and got dressed
     21 Agent: Who were you with?
     22 You:   (skipped)
     23 Agent: What did you use for it, such as tools, devices or a vehicle?
     24 You:   shower and clothes
     25 Agent: Where were you?
     26 You:   our shared bathroom
     27 Agent: How long did it take?
     28 You:   20 minutes
event3:
     29 Agent: What did you do at 9 o'clock?
     30 You:   I had breakfast with Lotte
     31 Agent: What did it involve? For example, what did you eat, make, read or talk abou

## Coreference chains

At the end of the conversation, the LLM groups the mentions of people in ```with_whom``` that refer to the same entity, e.g. ```his wife Gerda``` and ```Gerda```. Each entity has a so-called coreference chain with all the mentions of the same entity in the events. The user is the narrator and is mentioned in ```who``` in every event. Note that a conversation that was interrupted without **Stop** has no chains.

In [9]:
for mode, day in days.items():
    print(f"=== {mode}")
    for chain in day.get("coreference_chains", []):
        mentions = [f"{mention['event']}: {mention['mention']}" for mention in chain["mentions"]]
        print(f"{chain['entity_id']} ({chain['type']}, {chain['description']})")
        print("   ", "; ".join(mentions))

=== hour by hour
=== part of day
emma (person, the narrator)
    event1: Emma; event2: Emma; event3: Emma; event4: Emma; event5: Emma; event6: Emma; event7: Emma; event8: Emma; event9: Emma; event10: Emma; event11: Emma; event12: Emma; event13: Emma
lotte (person, A person related to Emma)
    event4: Lotte; event6: Lotte; event7: Lotte; event11: Lotte
suzie (person, A person related to Emma)
    event4: Suzie; event11: Suzie


## Comparing the two interviews

We now compare the two interviews with the statistics of ```day_statistics.py```. The two strategies are the two extremes of a scripted agent: hour by hour asks for every detail and takes many turns, per part of the day takes only three turns but only gets what the user happens to mention.

### Turns, events and roles

* **nr of turns**: the number of answers of the user about the events. The answers to the questions about the name, pronoun and home are not counted. With per part of the day, the events extracted from one answer share that turn, so we count the different turns.
* **nr of events**: the number of events in the day.
* **nr of roles**: the number of filled roles over all events (```who```, ```what```, ```with_what```, ```with_whom```, ```where```, ```when```, ```how_long```).
* **turn:event ratio**: the number of turns per event. Below 1 means that a single answer gave several events.
* **turn:role ratio**: the number of turns per filled role. The lower, the more efficient the interview.

In [10]:
def get_nr_of_turns(day):
    """The number of different answers of the user about the events of the day."""
    return len({turn["turn_id"] for event in get_events(day).values() for turn in get_turns(event, "human_turn")})

statistics = {mode: get_day_statistics(day) for mode, day in days.items()}

overview = {}
for mode, day in days.items():
    nr_of_turns = get_nr_of_turns(day)
    nr_of_events = statistics[mode]["nr_of_events"]
    nr_of_roles = statistics[mode]["nr_of_roles"]
    overview[mode] = {
        "nr of turns": nr_of_turns,
        "nr of events": nr_of_events,
        "nr of roles": nr_of_roles,
        "turn:event ratio": round(nr_of_turns / nr_of_events, 2) if nr_of_events else None,
        "turn:role ratio": round(nr_of_turns / nr_of_roles, 2) if nr_of_roles else None,
    }
# one row per interview, so that the counts stay integers and the ratios decimals
pd.DataFrame.from_dict(overview, orient="index")

,nr of turns,nr of events,nr of roles,turn:event ratio,turn:role ratio
hour by hour,23,5,30,4.60,0.77
part of day,3,13,55,0.23,0.05


### Role types

For each type of role, the number of events in which the role is filled (**roles**) and this number relative to the number of events of the interview (**per event**), i.e. the part of the events in which the role is filled. Few events give few roles, so the relative count shows how complete the events are, independent of how many events were collected. With per part of the day, the roles that the user did not mention stay empty, e.g. ```how_long``` when no times were given.

In [11]:
pd.DataFrame({(mode, count): {role: (roles if count == "roles" else
                                     round(roles / statistics[mode]["nr_of_events"], 2) if statistics[mode]["nr_of_events"] else None)
                              for role, roles in statistics[mode]["nr_of_roles_per_role"].items()}
              for mode in days for count in ["roles", "per event"]})

hour by hour           part of day          
                 roles per event       roles per event
who                  5       1.0          13      1.00
what                 3       0.6           5      0.38
with_what            5       1.0           6      0.46
with_whom            2       0.4           4      0.31
where                5       1.0          13      1.00
when                 5       1.0          13      1.00
how_long             5       1.0           1      0.08

### Role fillers

For each type of role, the number of values (**fillers**) over all events the number of different values (**unique**) and the number of values relative to the number of turns (**per turn**), i.e. how many fillers each answer of the user yielded on average. E.g. ```where``` has several fillers per event, from the room to the country, and the same ```who``` is the narrator in every event.

In [12]:
def get_filler_counts(day):
    """For each role, the number of fillers over all events, the number of unique fillers and the number of fillers
    per turn of the user."""
    events = get_events(day).values()
    nr_of_turns = get_nr_of_turns(day)
    counts = {}
    for role in ROLES:
        fillers = [str(value) for event in events for value in get_role_values(event, role)]
        counts[role] = {"fillers": len(fillers), "unique": len(set(fillers)),
                        "per turn": round(len(fillers) / nr_of_turns, 2) if nr_of_turns else None}
    return counts

filler_counts = {mode: get_filler_counts(day) for mode, day in days.items()}
pd.DataFrame({(mode, count): {role: counts[role][count] for role in ROLES}
              for mode, counts in filler_counts.items() for count in ["fillers", "unique", "per turn"]})

hour by hour                 part of day                
               fillers unique per turn     fillers unique per turn
who                  5      1     0.22          13      1     4.33
what                 9      9     0.39          10      9     3.33
with_what            8      8     0.35          11     11     3.67
with_whom            3      3     0.13           6      2     2.00
where               26     12     1.13          55     18    18.33
when                20     14     0.87          29     16     9.67
how_long             5      5     0.22           1      1     0.33

### The fillers of each role

For each role, the unique fillers with the number of times they are mentioned in the events of each interview, most frequent first. A filler that is missing in one of the interviews has 0 there. ```who``` and ```with_whom``` are combined, since both are people, and so are ```what``` and ```with_what```, since the LLM does not always agree with the user on whether something is the topic or the tool of an activity, e.g. a laptop.

In [13]:
ROLE_GROUPS = {"who & with_whom": ["who", "with_whom"], "what & with_what": ["what", "with_what"],
               "where": ["where"], "when": ["when"], "how_long": ["how_long"]}

def get_filler_frequencies(day, roles):
    """The unique fillers of the roles with the number of times they are mentioned over all events."""
    return Counter(str(value) for event in get_events(day).values() for role in roles
                   for value in get_role_values(event, role))

for group, roles in ROLE_GROUPS.items():
    fillers = pd.DataFrame({mode: get_filler_frequencies(day, roles) for mode, day in days.items()},
                           columns=list(days)).fillna(0).astype(int)
    fillers.index.name = group
    display(fillers.loc[fillers.sum(axis=1).sort_values(ascending=False, kind="stable").index])

,hour by hour,part of day
who & with_whom,,
Emma,5,13
Lotte,1,4
Suzie,0,2
flatmate Lotte,1,0
friend Suzie,1,0


,hour by hour,part of day
what & with_what,,
coffee,1,2
shower,2,0
clothes,2,0
yoghurt,1,1
fruit,1,1
bicycle,1,1
books,2,0
laptop,2,0
alarm clock,1,0


,hour by hour,part of day
where,,
Amsterdam,5,13
Europe,5,13
her house,3,8
bedroom of her house,1,4
Netherlands,5,0
kitchen,1,2
university,1,2
bedroom,1,1
bathroom,1,1


,hour by hour,part of day
when,,
morning,5,6
early in the morning,2,2
in the morning,2,2
afternoon,0,4
10 o'clock,1,2
evening,0,3
7 o'clock,1,1
08:00,1,1
8 o'clock,1,1


,hour by hour,part of day
how_long,,
30,1,1
5,1,0
20,1,0
45,1,0
60,1,0


### Coreferences

The number of mentions of each entity in the coreference chains, followed by the number of chains and the average number of mentions per chain. An entity that is missing in one of the interviews has 0 mentions there.

In [14]:
coreferences = pd.DataFrame({mode: statistics[mode]["nr_of_mentions_per_chain"] for mode in days}).fillna(0).astype(int)
coreferences.index.name = "entity"
display(coreferences)
pd.DataFrame.from_dict({mode: {"nr of chains": statistics[mode]["nr_of_coreference_chains"],
                               "average nr of mentions per chain": statistics[mode]["average_nr_of_mentions_per_chain"]}
                        for mode in days}, orient="index")

,hour by hour,part of day
entity,,
emma,0,13
lotte,0,4
suzie,0,2


,nr of chains,average nr of mentions per chain
hour by hour,0,0.00
part of day,3,6.33


## Using the agent in other code

The agent does not read or print anything itself: ```start()``` returns the first question and ```respond(utterance)``` takes the answer of the user and returns the next question. The conversation is finished when ```agent.done``` is ```True```. This makes it possible to connect the agent to other interfaces, such as a speech interface or the LEOLANI platform, e.g.:

```
agent = DayInterviewer(llama_client, model)
question = agent.start()
while not agent.done:
    answer = listen_to_the_user(question)
    question = agent.respond(answer)
```

The agent can also be run from a terminal in the root of the project:

```
python -m src.cltl.demo_chatbot.day_interviewer
```